In [ ]:
%pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [ ]:
%pip install google-auth==2.49.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.62.0
    Uninstalling google-auth-2.62.0:
      Successfully uninstalled google-auth-2.62.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.31.0 requires google-auth[requests]<3.0.0,>=2.56.0, but you have google-auth 2.49.0 which is incompatible.


In [ ]:
from google import genai

print("Gemini library is ready!")

Gemini library is ready!


In [ ]:
from google import genai
from google.colab import userdata

try:
    # Read the API key securely from Colab Secrets
    api_key = userdata.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError("API key not found in Colab Secrets.")

    # Connect to Gemini
    client = genai.Client(api_key=api_key)

    # Send your first question
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents="Introduce yourself in two simple sentences."
    )

    # Display the AI response
    print("Gemini says:")
    print(response.text)

except Exception as e:
    print("Something went wrong.")
    print("Error:", type(e).__name__, "-", str(e))

Something went wrong.
Error: ServerError - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with: My AI application is working!"
)

print(response.text)

My AI application is working!


In [ ]:
%pip install -q gradio

In [ ]:
import gradio as gr

print("Gradio is ready!")
print("Version:", gr.__version__)

Gradio is ready!
Version: 6.29.0


In [12]:
import gradio as gr
from google import genai
from google.genai import types
from google.colab import userdata

# Get API key securely
api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Check GEMINI_API_KEY in Colab Secrets.")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash-lite"


def chat(message, history, custom_instructions):
    message = (message or "").strip()
    history = history or []

    if not message:
        return history, ""

    try:
        # Keep conversation in Gradio messages format
        messages = []

        for item in history:
            if isinstance(item, dict):
                role = item.get("role")
                content = item.get("content", "")

                if role in ("user", "assistant") and isinstance(content, str):
                    messages.append({
                        "role": role,
                        "content": content[:1500]
                    })

            elif isinstance(item, (list, tuple)) and len(item) == 2:
                # Convert older pair-based history if present
                user_text, assistant_text = item

                if user_text:
                    messages.append({
                        "role": "user",
                        "content": str(user_text)[:1500]
                    })

                if assistant_text:
                    messages.append({
                        "role": "assistant",
                        "content": str(assistant_text)[:1500]
                    })

        # Use recent history to provide context
        recent_messages = messages[-10:]
        prompt = "\n".join(
            f"{'User' if item['role'] == 'user' else 'Assistant'}: "
            f"{item['content']}"
            for item in recent_messages
        )
        prompt += f"\nUser: {message}\nAssistant:"

        instructions = (custom_instructions or "").strip()

        if not instructions:
            instructions = "Answer clearly and use simple language."

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=instructions,
                temperature=0.7,
                max_output_tokens=512
            )
        )

        answer = response.text or "No response received."

    except Exception as e:
        error = str(e).upper()

        if "503" in error or "UNAVAILABLE" in error:
            answer = "Gemini is busy. Please try again shortly."
        elif "429" in error or "RESOURCE_EXHAUSTED" in error:
            answer = "API limit reached. Please check your quota."
        elif "401" in error or "403" in error:
            answer = "Authentication failed. Check your API key."
        else:
            answer = f"Request failed: {type(e).__name__}. Please try again."

    # IMPORTANT: Return dictionary messages, not [user, assistant] pairs
    updated_history = messages + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]

    return updated_history, ""


# Build the chat interface
with gr.Blocks(title="AI Productivity Chat") as demo:
    gr.Markdown("# 🤖 AI Productivity Chat")
    gr.Markdown("Chat with Gemini and customise its responses.")

    instructions = gr.Textbox(
        label="Custom Instructions",
        value="Explain everything in simple language with examples.",
        lines=2
    )

    chatbot = gr.Chatbot(
        label="Conversation",
        height=400
    )

    message = gr.Textbox(
        label="Your Message",
        placeholder="Type your question and press Enter...",
        lines=2
    )

    with gr.Row():
        send_button = gr.Button("Send 🚀")
        clear_button = gr.Button("Clear Chat")

    send_button.click(
        fn=chat,
        inputs=[message, chatbot, instructions],
        outputs=[chatbot, message]
    )

    message.submit(
        fn=chat,
        inputs=[message, chatbot, instructions],
        outputs=[chatbot, message]
    )

    clear_button.click(
        fn=lambda: ([], ""),
        inputs=None,
        outputs=[chatbot, message]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a1b9df18e9636d4d0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
from pydantic import BaseModel
from google.genai import types

# Define the required JSON format
class StudyGuide(BaseModel):
    topic: str
    difficulty: str
    key_points: list[str]

# Ask Gemini to follow this structure
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain Java OOP concepts for a beginner.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=StudyGuide,
        temperature=0.2,
        max_output_tokens=250
    )
)

# Validate and display the structured response
result = StudyGuide.model_validate_json(response.text)
print(result.model_dump_json(indent=2))

{
  "topic": "Java Object-Oriented Programming for Beginners",
  "difficulty": "Beginner",
  "key_points": [
    "Classes and Objects: A class is a blueprint, and an object is an instance of that class.",
    "Inheritance: Allows a class to inherit properties and methods from another class.",
    "Polymorphism: Enables objects to take on many forms, often achieved through method overriding and overloading.",
    "Encapsulation: Hiding internal state and requiring all interaction to be performed through an object's methods.",
    "Abstraction: Hiding complex implementation details and showing only the essential features."
  ]
}


In [14]:
from google.genai import types

prompt = "Explain Java inheritance in three simple points."

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.3,
        max_output_tokens=150
    )
)

print("AI Response:")
print(response.text)

# Display token usage
usage = response.usage_metadata

if usage:
    print("\n--- Token Usage Report ---")
    print("Input tokens:", usage.prompt_token_count)
    print("Output tokens:", usage.candidates_token_count)
    print("Total tokens:", usage.total_token_count)
else:
    print("Token usage information is unavailable.")

AI Response:
Here is Java inheritance in three simple points:

1. **Code Reusability:** It allows a new class (child/subclass) to reuse the fields and methods of an existing class (parent/superclass) so you don't have to rewrite the same code.
2. **The "IS-A" Relationship:** It represents a hierarchical relationship where the child class is a specialized version of the parent class (e.g., a `Dog` *is-a* `Animal`).
3. **Keyword `extends`:** You implement inheritance in Java simply by using the `extends` keyword in the child class definition (e.g., `class Dog extends Animal`).

--- Token Usage Report ---
Input tokens: 9
Output tokens: 140
Total tokens: 149


# Module 3: LLM APIs & Application Development

## Project Title
AI Chat Application Using Gemini API and Python

## Technologies Used
- Python
- Google Colab
- Gemini API
- Google GenAI Python SDK
- Gradio
- Pydantic for structured output

## Project Features
1. AI responses through the Gemini API.
2. Conversation history during the active chat session.
3. Custom instructions to control the AI's response style.
4. Structured JSON responses with schema validation.
5. Token usage tracking for input and output.
6. Error handling for API failures and quota limits.
7. A Clear Chat button to reset the conversation.

## Testing Report

| Test Case | Expected Result | Actual Result |
|---|---|---|
| Send a question | AI generates a response | Pass |
| Chat history | Previous messages remain visible | Pass |
| Custom instructions | AI follows the requested response style | Pass |
| Clear Chat | Conversation is cleared | Pass |
| Structured JSON | Output follows the defined schema | Record actual result |
| Token usage | Input, output and total tokens are displayed | Pass |
| API error handling | A helpful message appears when a request fails | Test before marking Pass |

## API Usage and Cost Management
- Limit the maximum response length.
- Use only the recent conversation history.
- Track input and output token counts.
- Check the current model pricing and account quota before estimating API costs.
- Keep the API key in Google Colab Secrets and never upload it to GitHub.

## Conclusion
This project demonstrates how to connect a Python application to an LLM API, maintain conversation context, customize AI responses, generate structured data, handle errors, and monitor token usage.

In [15]:
original_model = MODEL_NAME

try:
    # Use an intentionally invalid model for testing
    MODEL_NAME = "invalid-model-for-error-test"

    test_history, _ = chat(
        "Test error handling",
        [],
        "Answer briefly."
    )

    error_message = test_history[-1]["content"]
    print("Test result:")
    print(error_message)

    if error_message.startswith("Request failed:"):
        print("\nError handling test PASSED!")
    else:
        print("\nPlease check the result above.")

finally:
    # Restore your working model
    MODEL_NAME = original_model

Test result:
Request failed: ClientError. Please try again.

Error handling test PASSED!
